# 05. Colab GPU 모델 비교

먼저 별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 이 노트북을 실행합니다. 기준일 `t`까지 확인 가능한 정보로 `t+1`~`t+7`의 일별 판매량을 예측하고, LightGBM과 CatBoost를 같은 전체 이력 Feature 및 최근 28개 검증 기준일에서 비교합니다.

원본 데이터와 생성 파일은 Git에 올리지 않습니다. 전체 학습은 시간이 오래 걸리므로 필요한 셀만 실행하세요.


## Colab 준비


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/retail-demand-forecasting


### 패키지 설치

00 노트북에서 준비한 저장소의 의존성을 설치합니다. 이미 설치한 런타임이면 이 셀은 건너뛰어도 됩니다.


In [ ]:
%pip install -r requirements.txt -e .


### LightGBM CUDA 빌드

LightGBM CUDA 버전을 먼저 wheel로 빌드하고, 성공한 경우에만 설치합니다. 빌드가 실패해도 기존 LightGBM은 유지됩니다.


In [ ]:
%%bash
set -e
wheel_dir=$(mktemp -d)
python -m pip wheel --no-cache-dir --no-deps --no-binary=lightgbm --config-settings=cmake.define.USE_CUDA=ON --wheel-dir "$wheel_dir" lightgbm==4.7.0
python -m pip install --force-reinstall --no-deps "$wheel_dir"/lightgbm-4.7.0-*.whl


## Library


In [ ]:
import gc
import json
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

from retail_forecast.config import load_config
from retail_forecast.metrics import regression_metrics
from retail_forecast.pipeline import feature_cache_path, run_experiment


## Utils

실험 설정과 결과 표를 위한 보조 함수만 이곳에 둡니다. Feature 생성과 모델 학습은 `src/retail_forecast`의 함수를 사용합니다.


In [ ]:
def load_model_result(result_dir, model_name):
    metrics_path = result_dir / f"metrics_{model_name}.json"
    summary_path = result_dir / f"decision_summary_{model_name}.csv"
    horizon_path = result_dir / f"metrics_by_horizon_{model_name}.csv"
    missing = [path for path in (metrics_path, summary_path, horizon_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"모델 실행 결과가 없습니다: {missing[0]}")

    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary = pd.read_csv(summary_path, usecols=["actual_sum_1_7", "prediction_sum_1_7"])
    if summary.isna().any().any():
        raise ValueError(f"{model_name}: 7일 합계에 결측값이 있습니다")
    weekly = regression_metrics(
        summary["actual_sum_1_7"].to_numpy(),
        summary["prediction_sum_1_7"].to_numpy(),
    )
    row = {
        "model": model_name, "objective": metrics["objective"],
        "training_scope": "all_available_origins",
        "validation_origin_start": metrics["validation_origin_start"],
        "train_rows": metrics["train_rows"], "validation_rows": metrics["validation_rows"],
        "max_trees": metrics["model_params"].get("n_estimators", metrics["model_params"].get("iterations")),
        "trained_trees": metrics["trained_trees"],
        **{name: metrics[name] for name in ("rmsle", "mae", "rmse", "wape", "bias")},
        **{f"sum_1_7_{name}": value for name, value in weekly.items()},
    }
    horizon = pd.read_csv(horizon_path)
    horizon.insert(0, "model", model_name)
    horizon.insert(1, "objective", metrics["objective"])
    return row, horizon


def build_comparison_tables(result_dir):
    rows, horizons = [], []
    for model_name in ("lightgbm", "catboost"):
        row, horizon = load_model_result(result_dir, model_name)
        rows.append(row)
        horizons.append(horizon)
    return pd.DataFrame(rows), pd.concat(horizons, ignore_index=True)


def experiment_config(path, overrides, run_name):
    config = load_config(path)
    return replace(
        config,
        model_params={**config.model_params, **overrides},
        model_path=config.model_path.parent / run_name,
        output_path=config.output_path.parent / run_name,
    )


## 실험 파라미터

1200은 트리 깊이가 아니라 최대 생성 횟수입니다. 두 모델의 최대 횟수를 우선 600으로 제한하고 조기 종료를 유지합니다. LightGBM `max_depth=-1`은 깊이 제한 없음이며, CatBoost는 `depth=8`입니다. 다른 파라미터도 아래 모델별 딕셔너리에 추가하면 TOML 기본값을 덮어씁니다. 설정을 바꿔 다시 학습할 때는 `RUN_NAME`도 변경해 기존 결과를 보존하세요.


In [ ]:
REPO = Path.cwd()
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

RUN_NAME = "full_history_l2_i600"
MODEL_OVERRIDES = {
    "lightgbm": {"n_estimators": 600, "max_depth": -1, "early_stopping_rounds": 100},
    "catboost": {"iterations": 600, "depth": 8, "gpu_ram_part": 0.8, "early_stopping_rounds": 100},
}


## 비교 모델

| 모델 | 학습 방식 |
|---|---|
| LightGBM | 전체 이력 Feature 기반 GPU 학습 |
| CatBoost | 같은 Feature·검증 기준일의 GPU 학습 |

딥러닝 N-HiTS·TCN·TFT는 [07 Colab 노트북](07_dl_colab_validation.ipynb)에서 별도로 실행합니다.


## 1. Kaggle 원본과 전체 이력 Feature

Feature 캐시가 이미 있으면 이 셀은 건너뛰세요. 최초 생성 시 `train.csv`, `stores.csv`, `holidays_events.csv`, `transactions.csv`를 저장소의 `data/raw`에 넣습니다. 스크립트가 28일 단위 Parquet으로 저장하며, 완성된 캐시가 있으면 재생성하지 않습니다.


In [ ]:
!python scripts/build_full_features.py --config configs/full_lightgbm_gpu.toml --chunk-days 28


## 2. 모델별 전체 이력 학습

LightGBM은 L2(`regression`), CatBoost는 제곱오차 계열(`RMSE`)로 학습합니다. 두 설정은 같은 Feature 캐시와 검증 구간을 사용합니다. LightGBM 다음 CatBoost를 실행하세요. 결과는 `models/{RUN_NAME}`, `outputs/{RUN_NAME}`에 저장됩니다. 기존 결과를 보존하려면 실행 전에 `RUN_NAME`을 바꾸세요.

In [ ]:
lightgbm_config = experiment_config(
    REPO / "configs/full_lightgbm_gpu.toml", MODEL_OVERRIDES["lightgbm"], RUN_NAME,
)
if not feature_cache_path(lightgbm_config).exists():
    raise FileNotFoundError("전체 이력 Feature 캐시가 없습니다. 이전 셀을 먼저 실행하세요.")
lightgbm_result = run_experiment(lightgbm_config, progress=True)
print(lightgbm_result)

In [ ]:
gc.collect()
catboost_config = experiment_config(
    REPO / "configs/full_catboost_gpu.toml", MODEL_OVERRIDES["catboost"], RUN_NAME,
)
catboost_result = run_experiment(catboost_config, progress=True)
print(catboost_result)

## 3. 결과 표 저장

전체 일별 예측, horizon별 예측, 발주용 7일 합계의 오차를 별도 표로 저장합니다. 두 모델의 결과 파일이 모두 있을 때만 비교표를 만듭니다.

In [ ]:
result_dir = lightgbm_config.output_path
comparison, by_horizon = build_comparison_tables(result_dir)
comparison.to_csv(result_dir / "comparison_full_history.csv", index=False)
by_horizon.to_csv(result_dir / "comparison_by_horizon_full_history.csv", index=False)
display(comparison)
display(by_horizon)
print("저장 위치:", result_dir)


## 모델 비교 시각화


In [ ]:
plot_data = comparison.set_index("model")[["rmsle", "mae", "rmse", "wape"]]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, ("rmsle", "mae", "rmse", "wape")):
    plot_data[metric].sort_values().plot.barh(ax=ax, title=metric.upper())
    ax.set(xlabel="낮을수록 좋음", ylabel="")
plt.tight_layout()


## 모델 선택과 해석

WAPE와 7일 합계 오차를 중심으로 MAE, RMSE, RMSLE, Bias도 함께 봅니다. WAPE는 개별 판매량이 0이어도 계산할 수 있지만, 같은 검증 행에서는 MAE와 모델 순위가 같습니다. 성능 차이가 작다면 학습 시간과 설명 편의성을 고려합니다. 상품군·매장별 오류는 `06_error_analysis.ipynb`에서 분석합니다.

딥러닝 모델 결과는 같은 검증 기준일과 지표로 저장한 뒤 이 비교 표에 연결합니다. 아직 실행하지 않은 모델의 수치는 표시하지 않습니다.


### 해석과 주의사항

- 전체 이력은 *모든 기준일*을 사용한다는 뜻입니다. 마지막 28개 기준일의 목표 판매량은 학습에서 제외합니다.
- 이전 365일 결과와 학습 기간과 Feature 조건이 달라 숫자를 직접 비교하면 안 됩니다. 이번 설정은 예측 시점에 알 수 없는 지진·돌발 이벤트 변수를 제외합니다.
- 목표일 프로모션은 **7일 앞까지 계획값을 실제로 확보할 수 있을 때만** 유효합니다. 계획값이 없다면 두 설정에서 `target_onpromotion`을 제외하고 Feature 캐시를 그대로 사용해 재학습해야 합니다.
- `sales`와 `current_transactions`는 기준일 영업 종료 후 예측한다는 전제입니다. 영업 중 발주라면 이 변수도 제외해야 합니다.
- 전체 이력 표는 메모리를 많이 사용합니다. Colab의 GPU와 RAM 할당은 변할 수 있고, My Drive I/O는 느릴 수 있습니다. 런타임이 끊겨도 완성된 Parquet과 결과 파일은 Drive에 남습니다. `.building` 폴더가 남았다면 생성이 중단된 것이므로 내용 확인 후 수동으로 정리하세요.